# 1. import library

In [ ]:
import os
from glob import glob
from tqdm import tqdm

import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from scipy.spatial.distance import jensenshannon

from plotnine import *
import matplotlib.pyplot as plt
import seaborn as sns
from mizani.formatters import number_format

import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
from plotly.io import write_html
from matplotlib.colors import to_hex
import seaborn as sns
from svgutils.compose import Figure, SVG

# 2. import simulation_data

## 2.1. rawdata

In [ ]:
prefix = "../"
name_1 = "1_simulation/"
name_2 = "simu_result_continuous/"
name_3_1 = "history"
name_3_2 = "transition"

folder_name = os.path.join(prefix, name_1, name_2, name_3_1)
folder_name_2 = os.path.join(prefix, name_1, name_2, name_3_2)

json_files = sorted(glob(os.path.join(folder_name, "*.json")))
json_files_2 = sorted(glob(os.path.join(folder_name_2, "*.json")))

In [ ]:
# history data(individual cell data)
all_data = []
for file_path in tqdm(json_files, desc="Loading JSON files"):
    try:
        df = pd.read_json(file_path, lines=True)
        df["file_name"] = os.path.basename(file_path).replace(".json", "")
        all_data.append(df)
    except Exception as e:
        print(f"Failed to read {file_path}: {e}")

if all_data:
    simu_data = pd.concat(all_data, ignore_index=True)
else:
    raise ValueError("No valid JSON files found")

# transition data(biomass data)
all_data_2 = []
for file_path in tqdm(json_files_2, desc="Loading JSON files"):
    try:
        df = pd.read_json(file_path, lines=True)
        df["file_name"] = os.path.basename(file_path).replace(".json", "")
        all_data_2.append(df)
    except Exception as e:
        print(f"Failed to read {file_path}: {e}")

if all_data_2:
    simu_biomass_data = pd.concat(all_data_2, ignore_index=True)
else:
    raise ValueError("No valid JSON files found")

## 2.2. mutate

In [ ]:
# individual cell data
simu_data_mutate = simu_data.copy()

simu_data_mutate['Specie'] = 'PY1'
simu_data_mutate['Condition'] = 'no-supernatant'
simu_data_mutate['source'] = 'simu'
simu_data_mutate['deltaA'] = simu_data_mutate['Ad'] - simu_data_mutate['Ab']
simu_data_mutate['elongation_rate'] = simu_data_mutate['tau']

In [ ]:
# division_ratio data
simu_divR_data_mutate = simu_data.copy()

simu_divR_data_mutate['Specie'] = 'PY1'
simu_divR_data_mutate['Condition'] = 'no-supernatant'
simu_divR_data_mutate['source'] = 'simu'

In [ ]:
# biomass_transition data
simu_biomass_data_mutate = simu_biomass_data.copy()

simu_biomass_data_mutate['Specie'] = 'PY1'
simu_biomass_data_mutate['Condition'] = 'no-supernatant'
simu_biomass_data_mutate['source'] = 'simu'
simu_biomass_data_mutate['biomass_production_density'] = simu_biomass_data_mutate['biomass_production_density']
simu_biomass_data_mutate['group'] = simu_biomass_data_mutate["file_name"]

# 3. import experimental_data

## 3.1. rawdata

In [ ]:
prefix_2 = "../../1_fit_summarize_rawdata"
name_1_2 = "processed_data"
name_2_2 = "5_summary_data_exponential.csv"
name_2_3 = "6_daughter_cell_pair.csv"
name_2_4 = "3_biomass_data.csv"
file_name = os.path.join(prefix_2, name_1_2, name_2_2)
file_name_2 = os.path.join(prefix_2, name_1_2, name_2_3)
file_name_3 = os.path.join(prefix_2, name_1_2, name_2_4)

In [ ]:
exp_data = pd.read_csv(file_name)
exp_divR_data = pd.read_csv(file_name_2)
exp_biomass_data = pd.read_csv(file_name_3)

## 3.2. mutate

In [ ]:
# individual cell data
exp_data_mutate = exp_data.copy()
exp_data_mutate = exp_data_mutate.query('Specie == "PY1" and Condition == "no-supernatant"')

exp_data_mutate['id'] = exp_data_mutate['first_spot_ID']
exp_data_mutate['Ab'] = exp_data_mutate['fitted_Ab']
exp_data_mutate['Ad'] = exp_data_mutate['fitted_Ad']
exp_data_mutate['Tb'] = exp_data_mutate['Tb']
exp_data_mutate['age'] = exp_data_mutate['generation_time']
exp_data_mutate['elongation_rate'] = exp_data_mutate['fitted_elongation_rate']
exp_data_mutate['deltaA'] = exp_data_mutate['fitted_deltaA']
exp_data_mutate['generation_time'] = exp_data_mutate['generation_time']
exp_data_mutate['biomass_production_density_at0'] = exp_data_mutate['um3_biomass_production_density_start']
exp_data_mutate['file_name'] = np.nan
exp_data_mutate['dA_TH'] = np.nan
exp_data_mutate['source'] = 'exp'

In [ ]:
# division_ratio data
exp_divR_data_mutate = exp_divR_data.copy()
exp_divR_data_mutate = exp_divR_data_mutate.query('Specie == "PY1" and Condition == "no-supernatant"')

exp_divR_data_mutate['divR'] = exp_divR_data_mutate['div_ratio']
exp_divR_data_mutate['source'] = 'exp'

In [ ]:
# biomass_transition data
exp_biomass_data_mutate = exp_biomass_data.copy()
exp_biomass_data_mutate = exp_biomass_data_mutate.query('Specie == "PY1" and Condition == "no-supernatant"')

exp_biomass_data_mutate['source'] = 'exp'
exp_biomass_data_mutate['biomass_production_density'] = exp_biomass_data_mutate['um3_biomass_production_density']
exp_biomass_data_mutate['replicate'] = exp_biomass_data_mutate['replicate'].astype(str)
exp_biomass_data_mutate['FOV'] = exp_biomass_data_mutate['FOV'].astype(str)
exp_biomass_data_mutate['group'] = exp_biomass_data_mutate['replicate'] + '-' + exp_biomass_data_mutate['FOV']

# 4. concatenate simu_data and exp_data

## 4.1. concatenate

In [ ]:
# individual cell data
data = pd.concat((simu_data_mutate, exp_data_mutate))
data = data.loc[:, ["Specie", "Condition",
                    "id", "Ab", "Ad", "deltaA", 
                    "Tb", "age", "generation_time", 
                    "elongation_rate", "biomass_production_density_at0", 
                    "file_name", "dA_TH", "source"]]

In [ ]:
# division_ratio data
divR_data = pd.concat((simu_divR_data_mutate, exp_divR_data_mutate))
divR_data = divR_data.loc[:, ["Specie", "Condition", "divR", "source"]]

In [ ]:
# biomass_transiton data
biomass_data = pd.concat((simu_biomass_data_mutate, exp_biomass_data_mutate))
biomass_data = biomass_data.loc[:, ["Specie", "Condition", "group", "Time", "biomass_production_density", "source"]]

## 4.2. mutate

In [ ]:
specie_labels = {
    "PY1": "Nitrosomonas sp. PY1",
    "europaea": "N. europaea"
}
divide_labels = {
    "divide": "Dividing",
    "non-divide": "Non-dividing"
}
condition_labels = {
    "no-supernatant": "CFS -",
    "supernatant": "CFS +"
}
source_labels = {
    "simu": "\nsimulation data",
    "exp": "\nexperimental data"
}

data['Specie_label'] = data['Specie'].map(specie_labels).fillna(data['Specie'])
data['Condition_label'] = data['Condition'].map(condition_labels).fillna(data['Condition'])
data['source_label'] = data['source'].map(source_labels).fillna(data['source'])

divR_data['Specie_label'] = divR_data['Specie'].map(specie_labels).fillna(divR_data['Specie'])
divR_data['Condition_label'] = divR_data['Condition'].map(condition_labels).fillna(divR_data['Condition'])
divR_data['source_label'] = divR_data['source'].map(source_labels).fillna(divR_data['source'])

biomass_data['Specie_label'] = biomass_data['Specie'].map(specie_labels).fillna(biomass_data['Specie'])
biomass_data['Condition_label'] = biomass_data['Condition'].map(condition_labels).fillna(biomass_data['Condition'])
biomass_data['source_label'] = biomass_data['source'].map(source_labels).fillna(biomass_data['source'])

In [ ]:
data_G0 = data[data["Tb"] == 0]
data_Gn = data[data["Tb"] != 0]

data_obs = data_Gn[data_Gn["source_label"]=="\nexperimental data"]
data_sim = data_Gn[data_Gn["source_label"]=="\nsimulation data"]

# 5. plot_Gn

## 5.1. config

In [ ]:
# theme
mytheme_paper = theme(
    # axis
    text=element_text(family="Helvetica", size=8, color="black"),
    plot_title=element_text(size=9.5, weight='bold'),
    axis_title_x=element_text(size=8),
    axis_title_y=element_text(size=8),
    axis_text_x=element_text(size=6.5),
    axis_text_y=element_text(size=6.5),
    # axis_line=element_line(color="black", size=1.0),
    strip_text_x=element_text(size=8),
    strip_text_y=element_text(size=8),
    strip_background=element_blank(),

    # legend
    legend_position="none",
    legend_background=element_blank(),
    legend_title=element_blank(),
    legend_text=element_text(size=8),

    # panel
    panel_background=element_rect(fill="none"),
    panel_grid_major=element_line(color="none"),
    panel_grid_minor=element_line(color="none"),
    panel_border=element_rect(color="black", fill=None),

    # background
    plot_background=element_rect(fill="none")
)

plt.rcParams.update({
        "font.family": "Helvetica",
        "font.size": 8, 
        "axes.titlesize": 9.5,
        "axes.titleweight": "bold",
        "axes.labelsize": 8, 
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "legend.fontsize": 8,
        "axes.edgecolor": "black",
        "axes.linewidth": 1.0,
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "lines.markersize": 4.0,
    })

my_colors = {
    "PY1": "salmon",
    "europaea": "royalblue",
    "divide": "gray",
    "non-divide": "gold",
    "TRUE": "green",
    "FALSE": "gold",
    "exp": "salmon",
    "simu": "mistyrose",
}

In [ ]:
# save directory
output_dir = os.path.join("./result", "G_n")
os.makedirs(output_dir, exist_ok=True)
output_dir_jsd = os.path.join(output_dir, "JSD")
os.makedirs(output_dir_jsd, exist_ok=True)
output_dir_dynamics = os.path.join(output_dir, "single_cell_dynamics")
os.makedirs(output_dir_dynamics, exist_ok=True)
output_dir_jsd_map = os.path.join(output_dir, "JSD_heatmap")
os.makedirs(output_dir_jsd_map, exist_ok=True)

In [ ]:
x_col = 'biomass_production_density_at0'
y_col_list = ["generation_time", "elongation_rate",
              "deltaA", "Ab", "Ad"]
y_col_labels = [r"$\mathit{T}$", r"$\mathit{\alpha}$",
                r"$\mathit{\Delta A}$", r"$\mathit{Ab}$", r"$\mathit{Ad}$"]

plot_specs = {
    "generation_time": dict(
        y_label = r"Generation time,  $T$ (h)",
        y_scale = dict(breaks=np.arange(0, 120+1, 30), limits=(0, 120)),
        lm_label_scale = dict(x_limit=1.5*1e6, y_limit=115),
        show_limits = True
        ),
    "elongation_rate": dict(
        y_label = r"Elongation rate, $\alpha$ (h$^{-1}$)",
        y_scale = dict(breaks=np.arange(0, 0.12+1e-12, 0.03), limits=(0, 0.12)),
        lm_label_scale = dict(x_limit=1.5*1e6, y_limit=0.12),
        show_limits = False
        ),
    "deltaA": dict(
        y_label = r'$\Delta A$ ($\mu\mathrm{m}^2$)',
        y_scale = dict(breaks=np.arange(0, 2.5+1e-12, 0.5), limits=(0, 2.5)),
        lm_label_scale = dict(x_limit=1.5*1e6, y_limit=2.5),
        show_limits = False
        ),
    "Ab": dict(
        y_label = r"Area at birth, $Ab$ ($\mu\mathrm{m}^2$)",
        y_scale = dict(breaks=np.arange(0, 2.5+1e-12, 0.5), limits=(0, 2.5)),
        lm_label_scale = dict(x_limit=1.5*1e6, y_limit=2.5),
        show_limits = False
        ),
    "Ad": dict(
        y_label = r"Area at division, $Ad$ ($\mu\mathrm{m}^2$)",
        y_scale = dict(breaks=np.arange(0, 4.0+1e-12, 0.5), limits=(0, 4.0)),
        lm_label_scale = dict(x_limit=1.5*1e6, y_limit=4.0),
        show_limits = False
        )
}

JSD_dict = {}

## 5.2. functions

### 5.2.1. JSD functions

In [ ]:
def subsample_sim_by_fine_bins(
    sim_df, obs_df, x_col,
    n_bins=1000, min_target_to_sample=1, random_state=None
    ):

    # random generator
    rng = np.random.default_rng(random_state)

    # copy data
    sim = sim_df.copy().reset_index(drop=False)  # keep original index in 'index' column
    obs = obs_df.copy()

    # bin_edges
    x_min = obs[x_col].min()
    x_max = obs[x_col].max()
    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    # obs_counts per bin
    obs_counts, _ = np.histogram(obs[x_col].values, bins=bin_edges)
    # sim_counts per bin
    sim_bin_idx = np.digitize(sim[x_col].values, bins=bin_edges) - 1
    sim_bin_idx = np.clip(sim_bin_idx, 0, n_bins - 1)
    sim_counts = np.bincount(sim_bin_idx, minlength=n_bins)

    # iterate bins 
    chosen_idx_list = []
    chosen_counts = np.zeros(n_bins, dtype=int)
    sim_indices = sim["index"].values
    for b in range(n_bins):
        n_target = int(obs_counts[b])
        if n_target < min_target_to_sample:
            continue # skip if obs count is less than min_target_to_sample
        cand = sim_indices[sim_bin_idx == b]
        if cand.size < n_target:
            continue  # skip if not enough sim data
        picked = rng.choice(cand, size=n_target, replace=False)
        chosen_idx_list.extend(picked.tolist())
        chosen_counts[b] = n_target

    # build sim_subsampled df and diagnostics
    sim_subsampled = sim_df.loc[chosen_idx_list].reset_index(drop=True)
    diagnostics = {
        # "bin_edges": bin_edges,
        # "obs_counts_per_bin": obs_counts,
        # "sim_counts_per_bin": sim_counts,
        # "chosen_counts_per_bin": chosen_counts,
        "empty_obs_bins": int((obs_counts == 0).sum()),
        "n_bins": n_bins,
        "total_chosen": int(chosen_counts.sum()),
        "total_obs_nonzero_bins": int((obs_counts >= min_target_to_sample).sum()),
        "skipped_bins_due_to_sim_shortage": int(((obs_counts >= min_target_to_sample) & (sim_counts < obs_counts)).sum()),
    }

    return sim_subsampled, diagnostics

In [ ]:
def calculate_conditional_JSD(x_obs, y_obs, x_sim, y_sim,
                              K=50, y_bins=40, min_samples=10,
                              kde_bw_method="scott"):
    
    # Divide the x-axis into K segments
    x_all = np.concatenate([x_obs, x_sim])
    ps = np.linspace(0, 100, K+1)
    x_edges = np.percentile(x_all, ps)

    # To set the binning on the y-axis, configure the y-axis range.
    y_all = np.concatenate([y_obs, y_sim])
    y_min, y_max = np.percentile(y_all, [0.5, 99.5])
    span = y_max - y_min if (y_max > y_min) else 1.0
    y_min -= 0.05 * span
    y_max += 0.05 * span
    y_grid = np.linspace(y_min, y_max, y_bins)  # Set evaluation points for y_range for KDE implementation
    y_edges = np.linspace(y_min, y_max, y_bins+1) # Set the boundary line when binning y_range
    
    per_bin_jsd = []
    bin_centers = []
    counts_obs = []
    counts_sim = []

    # Calculate the JSD for each bin from Bin 0 to Bin K.
    for i in range(K):
        lo, hi = x_edges[i], x_edges[i+1]
        if i < K-1:
            mask_obs = (x_obs >= lo) & (x_obs < hi)
            mask_sim = (x_sim >= lo) & (x_sim < hi)
        else:
            mask_obs = (x_obs >= lo) & (x_obs <= hi)
            mask_sim = (x_sim >= lo) & (x_sim <= hi)

        y_obs_bin = y_obs[mask_obs]
        y_sim_bin = y_sim[mask_sim]

        counts_obs.append(len(y_obs_bin))
        counts_sim.append(len(y_sim_bin))
        bin_centers.append((lo + hi) / 2.0)

        if len(y_obs_bin) < min_samples or len(y_sim_bin) < min_samples:
            per_bin_jsd.append(np.nan)
            continue

        try:
            # Reproduce the distribution of binned data using KDE
            kde_p = gaussian_kde(y_obs_bin, bw_method=kde_bw_method)
            kde_q = gaussian_kde(y_sim_bin, bw_method=kde_bw_method)
            # Calculate the density at the evaluation point (y_grid)
            p_vals = kde_p(y_grid)
            q_vals = kde_q(y_grid)
            # Clip less than or equal to zero and add eps.
            p = p_vals.clip(min=0) + 1e-12
            q = q_vals.clip(min=0) + 1e-12
            # normalize
            p /= p.sum()
            q /= q.sum()
        except Exception as e:
            print(f"KDE failed: {e}.")
            per_bin_jsd.append(np.nan)
            continue

        # compute distances
        jsd = jensenshannon(p, q, base=2.0)
        per_bin_jsd.append(float(jsd))

    per_bin_jsd = np.array(per_bin_jsd)
    bin_centers = np.array(bin_centers)
    counts_obs = np.array(counts_obs)
    counts_sim = np.array(counts_sim)

    results = {
        "x_edges": x_edges,
        "y_edges": y_edges,
        "y_grid": y_grid,
        "bin_centers": bin_centers,
        "per_bin_jsd": per_bin_jsd,
        "counts_obs": counts_obs,
        "counts_sim": counts_sim,
        "kde_bw_method": kde_bw_method
    }
    return results

In [ ]:
def jsd_with_repeated_subsampling(
    x_obs, y_obs, source_obs, x_sim, y_sim, source_sim,
    x_col, y_col,
    n_repeats=30, base_seed=0,
    agg="mean"  # "mean" or "median"
):
    jsd_list = []
    res_rep = None
    sim_sub_pd_rep = None

    for r in range(n_repeats):
        seed = base_seed + r

        # --- prepare dataframes ---
        sim_df_pd = pd.DataFrame({
            x_col: x_sim,
            y_col: y_sim,
            "source_label": source_sim
        })
        obs_df_pd = pd.DataFrame({
            x_col: x_obs,
            y_col: y_obs,
            "source_label": source_obs
        })

        # --- subsample ---
        sim_sub_pd, diag = subsample_sim_by_fine_bins(
            sim_df_pd, obs_df_pd, x_col,
            n_bins=100, min_target_to_sample=1, random_state=seed
            )
        if sim_sub_pd_rep is None:
            sim_sub_pd_rep = sim_sub_pd  # save reprsentative

        # --- extract numpy arrays ---
        x_sim_sub = sim_sub_pd[x_col].to_numpy()
        y_sim_sub = sim_sub_pd[y_col].to_numpy()
        # ---- JSD per bin ----
        res = calculate_conditional_JSD(
            x_obs, y_obs, x_sim_sub, y_sim_sub,
            K=10, y_bins=100, min_samples=10, 
            kde_bw_method="scott"
        )
        if res_rep is None:
            res_rep = res  # save reprsentative
            
        jsd_list.append(res["per_bin_jsd"])

    jsd_mat = np.vstack(jsd_list)  # shape (n_repeats, K)

    # ---- aggregate across repeats (ignore NaN) ----
    if agg == "mean":
        per_bin_jsd_agg = np.nanmean(jsd_mat, axis=0)
    elif agg == "median":
        per_bin_jsd_agg = np.nanmedian(jsd_mat, axis=0)
    else:
        raise ValueError("agg must be 'mean' or 'median'")

    # --- return agg ---
    res_agg = dict(res_rep)  # shallow copy
    res_agg["per_bin_jsd"] = per_bin_jsd_agg
    res_agg["jsd_repeats"] = jsd_mat
    res_agg["subsampling"] = {"n_repeats": n_repeats, "base_seed": base_seed, "agg": agg}

    return sim_sub_pd_rep, obs_df_pd, res_rep, res_agg, jsd_mat

### 5.2.2. plot functions

In [ ]:
def plot_conditional_JSD(x_obs, y_obs, x_sim, y_sim, 
                         results, output_dir, name, parameter_name,
                         show_bins=[0,1,2,-1], use_kde=True):
    bin_centers = results["bin_centers"]
    per_bin_jsd = results["per_bin_jsd"]
    counts_obs = results["counts_obs"]
    counts_sim = results["counts_sim"]

    # --- JSD vs X ---
    fig, ax = plt.subplots(1, 1, 
                           figsize=(3.2,2.4), 
                           constrained_layout=True)
    ax.plot(bin_centers, per_bin_jsd, marker='o', label='')
    ax.set_ylabel('Jensen-Shannon divergence')
    ax.set_xlabel(r'$\Delta V_t$ ($\times 10^6\ \mu\mathrm{m}^3\,\mathrm{mL}^{-1}$)')
    ax.set_title(f'{parameter_name}')
    
    name_1 = f"JSD_vs_X_{name}.pdf"
    fig.savefig(os.path.join(output_dir, name_1),
                dpi=600, transparent=True)

    plt.tight_layout()
    # plt.show()
    plt.close(fig)

    # --- Example Y distributions in specific X-bins ---
    y_edges = results["y_edges"]
    x_edges = results["x_edges"]
    fig, axes = plt.subplots(2, 2, 
                             figsize=(6.4, 4.8),
                             sharex=True, 
                             sharey=True, 
                             constrained_layout=True)
    axes = axes.flatten()

    for i, bi in enumerate(show_bins):
        if bi < 0:
            bi = len(results["bin_centers"]) + bi
        if bi < 0 or bi >= len(results["bin_centers"]):
            print("error")
            continue
        ax = axes[i]
        lo, hi = x_edges[bi], x_edges[bi+1]
        print(f"Bin {bi}: ∆Vt in [{lo:.3g}, {hi:.3g}], counts obs={counts_obs[bi]}, sim={counts_sim[bi]}")

        mask_obs = (x_obs >= lo) & (x_obs < hi) if bi < (len(x_edges)-2) else (x_obs >= lo) & (x_obs <= hi)
        mask_sim = (x_sim >= lo) & (x_sim < hi) if bi < (len(x_edges)-2) else (x_sim >= lo) & (x_sim <= hi)
        y_obs_bin = y_obs[mask_obs]
        y_sim_bin = y_sim[mask_sim]

        # histogram
        ax.hist(y_obs_bin, bins=y_edges, density=True, alpha=0.5, label='Observed (hist)')
        ax.hist(y_sim_bin, bins=y_edges, density=True, alpha=0.4, label='Simulated (hist)')

        # KDE
        if use_kde and len(y_obs_bin) > 1 and len(y_sim_bin) > 1:
            y_grid = np.linspace(min(y_edges), max(y_edges), 200)
            kde_obs = gaussian_kde(y_obs_bin)
            kde_sim = gaussian_kde(y_sim_bin)
            ax.plot(y_grid, kde_obs(y_grid), color='C0', lw=1, label='Observed (KDE)')
            ax.plot(y_grid, kde_sim(y_grid), color='C1', lw=1, label='Simulated (KDE)')

        ax.set_title(fr"$\Delta V_t$-bin {bi}")
        ax.set_xlabel(parameter_name)
        ax.set_ylabel("Density")
        ax.legend(facecolor='none', edgecolor='none')
    
    name_2 = f"Binned_Y_dist_{name}.pdf"    
    fig.savefig(os.path.join(output_dir, name_2),
                dpi=600, transparent=True)
        
    plt.tight_layout()
    # plt.show()
    plt.close(fig)

In [ ]:
def calc_limits(df, y_col):
    y_min = df[y_col].min()
    y_max = df[y_col].max()
    
    return pd.Series({'y_min': y_min, 'y_max': y_max})

def get_lm_label_T(df, x_col, y_col, x_limit, y_limit):
    r = df[y_col].corr(df[x_col])
    y_min = df[y_col].min()
    y_max = df[y_col].max()
    
    label = (
        f"Max = {round(y_max, 1)}"
        f"\nMin = {round(y_min, 1)}"
        f"\nr = {r:.2f}"
        )
    
    x_pos = x_limit *0.95
    y_pos = y_limit *0.95
    
    return pd.Series({'label': label, 'x_pos': x_pos, 'y_pos': y_pos})

def get_lm_label(df, x_col, y_col, x_limit, y_limit):
    r = df[y_col].corr(df[x_col])
    
    label = (
        f"r = {r:.2f}"
        )

    x_pos = x_limit *0.95
    y_pos = y_limit *0.95
    
    
    return pd.Series({'label': label, 'x_pos': x_pos, 'y_pos': y_pos})

In [ ]:
def plot_single_cell_dynamics(obs_df_pd, sim_sub_pd_rep, 
                              spec, x_col, y_col,
                              output_dir):
    merged_data = pd.concat([obs_df_pd, sim_sub_pd_rep], axis=0)
    
    # --- labels ---
    if spec["show_limits"]:
        lm_label = merged_data.groupby('source_label').apply(
            get_lm_label_T, x_col=x_col, y_col=y_col,
            x_limit=spec["lm_label_scale"]["x_limit"], 
            y_limit=spec["lm_label_scale"]["y_limit"],
            include_groups=False
        ).reset_index()
    else:
        lm_label = merged_data.groupby('source_label').apply(
            get_lm_label, x_col=x_col, y_col=y_col,
            x_limit=spec["lm_label_scale"]["x_limit"], 
            y_limit=spec["lm_label_scale"]["y_limit"],
            include_groups=False
        ).reset_index()
        

    # --- plot ---
    g = (
        ggplot()
        + geom_point(data=merged_data, 
                    mapping=aes(x=x_col, y=y_col),
                    fill = "salmon",
                    shape='o', alpha=0.5, size=2)
        
        # --- linear regression + label ---
        + geom_smooth(data = merged_data,
                      mapping=aes(x=x_col, y=y_col),
                      method = "lm", se = False, color = "black")
        + geom_text(data=lm_label,
                    mapping=aes(x='x_pos', y='y_pos', label='label'),
                    ha='right',
                    va='top',
                    size=8)
        
        + scale_fill_manual(values = my_colors)
        + scale_x_continuous(breaks= np.arange(0, 1.5*1e6+1, 0.5*1e6),
                            limits=(0, 1.5*1e6),
                            labels=number_format(scale=1e-6)
                            )
        + scale_y_continuous(
            breaks=spec["y_scale"]["breaks"],
            limits=spec["y_scale"]["limits"]
            )
        + labs(
            x=r'$\Delta V_t$ ($\times 10^6\ \mu\mathrm{m}^3\,\mathrm{mL}^{-1}$)',
            y=spec["y_label"])
        + facet_grid(". ~ source_label")
        + mytheme_paper
    )

    # ---- max_val, min_val ----
    if spec["show_limits"]:
        y_limits = merged_data.groupby('source_label').apply(
                calc_limits, y_col=y_col, include_groups=False
            ).reset_index()
        g = (
            g
            + geom_hline(
                data=y_limits,
                mapping=aes(yintercept="y_max"),
                linetype="dashed"
            )
            + geom_hline(
                data=y_limits,
                mapping=aes(yintercept="y_min"),
                linetype="dashed"
            )
        )

    g.save(os.path.join(output_dir, f"{y_col}.pdf"), 
           dpi=600, width=3.0*2, height=2.4, transparent=True,
           facecolor="none", edgecolor="none")
    g.save(os.path.join(output_dir, f"{y_col}.svg"),
           width=3.0*2, height=2.4, transparent=True,
           facecolor="none", edgecolor="none")

In [ ]:
def plot_jsd_heatmap(jsd_dict, x_bin_centers, 
                     output_dir, name="JSD_heatmap", IF_annotate=False):
    param_names = list(jsd_dict.keys())
    jsd_matrix = np.array([jsd_dict[param] for param in param_names])

    # --- heatmap ---
    fig, ax = plt.subplots(figsize=(3.2, 2.4))
    sns.heatmap(jsd_matrix, 
                annot=IF_annotate, fmt=".2f", 
                cmap="magma",
                vmin=0, vmax=1,
                xticklabels=[f"{x/1e6:.2f}" for x in x_bin_centers],
                yticklabels=param_names, 
                annot_kws={"size": 4.0},
                ax=ax,
                linewidths=0.1, linecolor='white'
                )
    
    # --- color bar ---
    cbar = ax.collections[0].colorbar
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cbar.set_label("Jensen–Shannon divergence")

    ax.set_xlabel(r'$\Delta V_t$ ($\times 10^6\ \mu\mathrm{m}^3\,\mathrm{mL}^{-1}$)')
    fig.text(0.05, 0.95, "C", fontweight='bold', fontsize = 12)

    plt.tight_layout()
    ax.set_facecolor("none")
    fig.patch.set_alpha(0)

    # --- save ---
    if IF_annotate == False: 
        fig.savefig(os.path.join(output_dir, name + "_noAnnot.pdf"), 
                    dpi=600, transparent=True)
        fig.savefig(os.path.join(output_dir, name + "_noAnnot.svg"), 
                    dpi=600, transparent=True)
        plt.close(fig)
    
    if IF_annotate == True: 
        fig.savefig(os.path.join(output_dir, name + "_annot.pdf"), 
                    dpi=600, transparent=True)
        fig.savefig(os.path.join(output_dir, name + "_annot.svg"), 
                    dpi=600, transparent=True)
        plt.close(fig)
    
    save_path = os.path.join(output_dir, name + ".pdf & .svg")
    print(f"Heatmap saved to {save_path}")

## 5.3.  Plot JSD and dynamics

In [ ]:
# JSD and single-cell dynamics
for y_col, y_col_label in zip(y_col_list, y_col_labels):
    x_obs = data_obs[x_col].to_numpy()
    y_obs = data_obs[y_col].to_numpy()
    source_obs = data_obs["source_label"].to_numpy()
    x_sim = data_sim[x_col].to_numpy()
    y_sim = data_sim[y_col].to_numpy()
    source_sim = data_sim["source_label"].to_numpy()

    sim_sub_pd_rep, obs_df_pd, res_rep, res_agg, js_mat = jsd_with_repeated_subsampling(
        x_obs, y_obs, source_obs, x_sim, y_sim, source_sim, 
        x_col, y_col,
        n_repeats=30, base_seed=0,
        agg="mean"
    )

    # --- plot ---
    plot_conditional_JSD(
        x_obs, y_obs, x_sim, y_sim,
        res_rep,
        output_dir_jsd,
        name=f"{y_col}",
        parameter_name=y_col_label,
        show_bins=[0,1,2,-1],
        use_kde=True
    )

    plot_single_cell_dynamics(
        obs_df_pd,
        sim_sub_pd_rep,
        spec=plot_specs[y_col],
        x_col=x_col,
        y_col=y_col,
        output_dir=output_dir_dynamics
    )

    # --- data for heatmap ---
    JSD_dict[y_col_label] = res_agg["per_bin_jsd"]

In [ ]:
# JSD heatmap
x_bin_centers = res_rep["bin_centers"]

plot_jsd_heatmap(JSD_dict, x_bin_centers, 
                 output_dir_jsd_map, IF_annotate = False)
plot_jsd_heatmap(JSD_dict, x_bin_centers, 
                 output_dir_jsd_map, IF_annotate = True)

# 6. plot_G0

## 6.1. config.

In [ ]:
x_col_list_G0 = ['age', 'elongation_rate', 'deltaA', 'Ab', 'Ad']
x_col_labels_G0 = [r"$\mathit{T}$", r"$\mathit{\alpha}$",
                   r"$\mathit{\Delta A}$", r"$\mathit{Ab}$", r"$\mathit{Ad}$"]

output_dir = os.path.join("./result", "G_0")
os.makedirs(output_dir, exist_ok=True)

## 6.2. functions

In [ ]:
def plot_histogram(data, x_col, x_col_label, output_dir):
    g = (
        ggplot(data,
               aes(x=x_col, y='..density..', fill='Specie'))
        + geom_histogram(
            bins=20,
            alpha=0.5,
            color="black"
        )
        + scale_fill_manual(values = my_colors)
        + labs(x=x_col_label,
               y='Density')
        + facet_grid(". ~ source_label")
        + mytheme_paper
    )

    g.save(os.path.join(output_dir, f"{x_col}.pdf"), 
           dpi=600, width=3.2*2, height=2.4*1.1, transparent=True)
    g.save(os.path.join(output_dir, f"{x_col}.svg"),
           width=3.2*2, height=2.4*1.1, transparent=True)

## 6.3. plot

In [ ]:
for x_col, x_col_label in zip(x_col_list_G0, x_col_labels_G0):
    plot_histogram(data_G0, x_col, x_col_label, output_dir)

In [ ]:
# --- color ---
palette = sns.color_palette("Set2", len(data_G0["source_label"].unique()))
color_map = dict(zip(data_G0["source_label"].unique(), [to_hex(c) for c in palette]))

# --- plot ---
fig = go.Figure()

for src in data_G0["source_label"].unique():
    sub_src = data_G0[data_G0["source_label"] == src]
    for sp in sub_src["Specie"].unique():
        sub = sub_src[sub_src["Specie"] == sp]
        fig.add_trace(go.Scatter3d(
            x=sub["age"],
            y=sub["elongation_rate"],
            z=sub["Ab"],
            mode="markers",
            marker=dict(
                size=5,
                color=color_map[src],
                opacity=0.8,
                line=dict(width=0.5, color="black")
            ),
            name=f"{src} - {sp}",
            text=sub["Specie"],
            hovertemplate=(
                "source_label: " + src +
                "<br>Specie: %{text}"
                "<br>Age: %{x:.2f} h"
                "<br>Generation time: %{y:.2f} h"
                "<br>Area at birth: %{z:.2f} µm²<extra></extra>"
            )
        ))

# --- layout ---
fig.update_layout(
    title="3D Scatter plot by source_label × Specie",
    scene=dict(
        xaxis_title="Generation time, T (h)",
        yaxis_title="Elongation rate, α (h⁻¹)",
        zaxis_title="Area at birth, Ab (µm²)",
    ),
    legend_title="source_label - Specie",
    height=700,
    margin=dict(l=0, r=0, t=50, b=0),
)

# --- display ---
# fig.show()

# --- save ---
output_path = os.path.join(output_dir, "T_vs_alpha_vs_Ab.html")
fig.write_html(output_path, include_plotlyjs="cdn", auto_open=False)

# 7. other plots

In [ ]:
output_dir = os.path.join("./result", "other_plots")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
plot_histogram(divR_data, 'divR', r"$DR$", output_dir)

In [ ]:
# --- ∆Vt vs time plot ---
delta_Vt_vs_time_plot = (
    ggplot(biomass_data, 
           aes(x='Time', 
               y='biomass_production_density', 
               fill='Specie', color = 'Specie',group='group'))
    + geom_line(size=0.5, linetype='-')
    + scale_fill_manual(values = my_colors)
    + scale_color_manual(values = my_colors)
    + scale_y_continuous(breaks= np.arange(0, 1.5*1e6+1, 0.5*1e6),
                         limits=(0, 1.5*1e6),
                         labels=number_format(scale=1e-6)
                         )
    + scale_x_continuous(
           breaks=np.arange(0, 121, 40),
           limits=(0, 120)
           )
    + labs(
           x="Time (h)\n", 
           y=r'$\Delta V_t$ ($\times 10^6\ \mu\mathrm{m}^3\,\mathrm{mL}^{-1}$)',
           )
    + facet_grid(". ~ source_label")
    + mytheme_paper
)

# --- save ---
delta_Vt_vs_time_plot.save(os.path.join(output_dir, f"delta_Vt_vs_time_plot.pdf"), 
                           dpi=600, width=3.2*2, height=2.4*1.1, transparent=True)
delta_Vt_vs_time_plot.save(os.path.join(output_dir, f"delta_Vt_vs_time_plot.svg"), 
                           width=3.2*2, height=2.4*1.1, transparent=True)

# --- add panel label ---
fig = delta_Vt_vs_time_plot.draw()
fig.set_size_inches(3.2, 2.4)
fig.text(0.025, 1.0, "B", fontweight='bold', fontsize=12, va='top')

# --- save ---
fig.savefig(os.path.join(output_dir, f"delta_Vt_vs_time_plot_tagged.pdf"), 
            dpi=600, transparent=True)
fig.savefig(os.path.join(output_dir, f"delta_Vt_vs_time_plot_tagged.svg"), 
            transparent=True)
plt.close(fig)

# 8. Concatenate SVG

In [ ]:
prefix = "./result"
folder = "G_n/single_cell_dynamics"
pt_to_mm = 25.4 / 72

Figure(
    "160mm", "310mm",
    SVG(os.path.join(prefix, folder, "generation_time.svg")).scale(pt_to_mm).move(0, 0),
    SVG(os.path.join(prefix, folder, "elongation_rate.svg")).scale(pt_to_mm).move(0, 2.4*25.4),
    SVG(os.path.join(prefix, folder, "deltaA.svg")).scale(pt_to_mm).move(0, 2.4*25.4*2),
    SVG(os.path.join(prefix, folder, "Ab.svg")).scale(pt_to_mm).move(0, 2.4*25.4*3),
    SVG(os.path.join(prefix, folder, "Ad.svg")).scale(pt_to_mm).move(0, 2.4*25.4*4)
).save(os.path.join(prefix, "Figure_S11.svg"))